# Track B — Fase 3: Retrain di folds_v2 (data bersih)

**BDC Satria Data 2026** | Embedding TIDAK diekstrak ulang — cuma reindex ke baris yang tersisa di `folds_v2.csv`, lalu CV ulang.

| | |
|---|---|
| Komposisi terkunci | `siglip2so400m` (frozen) + kNN (k=15, cosine, weights=distance), **non-TTA** |
| Referensi v1 | CV mean=0.9901, min=0.9896, std=0.0005 |
| Sumber split | **kolom `fold` dari folds_v2** (dipakai apa adanya, TIDAK di-stratify ulang) |
| Output | `oof_siglip2so400m_knn_v2.npy` + meta → Track C |

> **Kenapa murah:** embedding gambar sudah jadi (`emb_*.npy` di Drive). Cleaning Track A cuma *drop* + *relabel* baris. Jadi kita ambil matriks embedding v1, susun ulang barisnya mengikuti `folds_v2` (join lewat `filepath`), pakai label & fold dari v2, lalu jalankan CV yang sama.

> **Kenapa join lewat `filepath`, bukan index:** embedding v1 selaras dengan **urutan baris `folds.csv` v1** (kontrak: baris ke-i embedding = baris ke-i folds v1). folds_v2 sudah beda panjang & urutan, jadi cara paling aman memetakan tiap baris v2 → posisinya di v1 adalah lewat `filepath`. Kalau format path Track A berubah sedikit pun, assert di bawah akan gagal keras — bukan diam-diam salah align.

> **Paritas metodologi:** CV di sini pakai hyperparameter terkunci + metrik yang sama (`labels=[0,1,2]`, `zero_division=0.0`) + kolom fold yang sama, jadi v2 bisa dibandingkan langsung dengan v1 di level per-fold.

---
## 🔧 SETUP

In [ ]:
# Cell 1 — Repo + Drive + dependensi (CPU, tanpa GPU)
import os
if not os.path.exists('/content/satria-data-bdcugm02'):
    !git clone https://github.com/agaggigit/satria-data-bdcugm02.git
else:
    !git -C /content/satria-data-bdcugm02 pull

!pip install -q scikit-learn

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✅ setup siap')

In [ ]:
# Cell 2 — Path dari CFG + verifikasi file WAJIB ada sebelum mulai
import os, sys
sys.path.insert(0, '/content/satria-data-bdcugm02/track_b/src')
from config import CFG

LOCKED   = 'siglip2so400m'                 # backbone terkunci (so400m-patch14-384), non-TTA
EMB      = CFG.embeddings_dir
FOLDS_V1 = CFG.folds_csv                    # v1 = urutan yang dipakai saat ekstraksi embedding
FOLDS_V2 = os.path.join(os.path.dirname(CFG.folds_csv), 'folds_v2.csv')
EMB_TRAIN = os.path.join(EMB, f'{LOCKED}_train.npy')

checks = [(FOLDS_V1,'folds v1'), (FOLDS_V2,'folds v2'), (EMB_TRAIN,'emb train terkunci')]
ok = True
for p, tag in checks:
    exists = os.path.exists(p)
    ok = ok and exists
    print(('✅' if exists else '❌ HILANG'), f'{tag:20s}', p)
assert ok, 'Ada file wajib yang hilang — selesaikan dulu sebelum retrain'
print('seed =', CFG.seed)

---
## 🔗 ALIGN — reindex embedding v1 → urutan folds_v2 (join lewat filepath)

In [ ]:
# Cell 3 — Bangun pemetaan filepath → posisi di v1, lalu diagnosa drop/relabel
import pandas as pd, numpy as np

v1 = pd.read_csv(FOLDS_V1)
v2 = pd.read_csv(FOLDS_V2)

for col in ['filepath', 'label', 'fold']:
    assert col in v1.columns, f'kolom {col!r} tak ada di folds v1'
    assert col in v2.columns, f'kolom {col!r} tak ada di folds v2'

# filepath harus unik di kedua sisi — kalau tidak, join ambigu
assert v1['filepath'].is_unique, 'filepath v1 tidak unik — join tak bisa dipercaya'
assert v2['filepath'].is_unique, 'filepath v2 tidak unik — cek Track A'

pos_of = {fp: i for i, fp in enumerate(v1['filepath'])}

# Setiap baris v2 WAJIB punya asal di v1 (cleaning cuma drop/relabel, tak menambah data)
missing = [fp for fp in v2['filepath'] if fp not in pos_of]
assert not missing, (f'{len(missing)} filepath di v2 tak ada di v1 — join filepath gagal. '
                     f'Contoh: {missing[:3]}. Cek apakah Track A mengubah format path.')

pos = v2['filepath'].map(pos_of).to_numpy()          # posisi tiap baris v2 di dalam matriks embedding v1

# --- Diagnostik (masuk report: berapa yang dibuang & dilabel-ulang) ---
v1_label = dict(zip(v1['filepath'], v1['label']))
v1_fold  = dict(zip(v1['filepath'], v1['fold']))

n_drop     = len(v1) - len(v2)
relabeled  = int(sum(v1_label[fp] != lb for fp, lb in zip(v2['filepath'], v2['label'])))
fold_moved = int(sum(v1_fold[fp]  != fd for fp, fd in zip(v2['filepath'], v2['fold'])))

dropped_fp = set(v1['filepath']) - set(v2['filepath'])
drop_by_cls = v1[v1['filepath'].isin(dropped_fp)]['label'].value_counts().sort_index()

print(f'v1: {len(v1)} baris   v2: {len(v2)} baris')
print(f'DROP  : {n_drop}  ({n_drop/len(v1)*100:.2f}% dari train)')
print(f'RELABEL: {relabeled}')
print('drop per kelas (0=Recyclable,1=Electronic,2=Organic):')
print(drop_by_cls.to_string())

# Aturan keras Track A: fold sampel yang BERTAHAN tidak boleh berubah,
# kalau berubah v2 tak sebanding v1 (OOF beda basis).
assert fold_moved == 0, (f'{fold_moved} sampel survivor pindah fold — langgar aturan Track A. '
                         f'v2 tak bisa dibandingkan apple-to-apple dgn v1. STOP, klarifikasi ke Track A.')
print('✅ fold survivor tak berubah — v2 sebanding dgn v1')

# Rem pengaman: kalau drop > 3%, kemungkinan cleaning terlalu agresif (cek plan Track A cap 2-3%)
if n_drop / len(v1) > 0.03:
    print(f'⚠️  DROP {n_drop/len(v1)*100:.2f}% > cap 3% — konfirmasi ke Track A sebelum lanjut')

In [ ]:
# Cell 4 — Load embedding v1, susun ulang ke urutan v2, siapkan y & fold
X1 = np.load(EMB_TRAIN)
assert len(X1) == len(v1), ('embedding v1 tak sepanjang folds v1 — kontrak alignment putus. '
                            f'emb={len(X1)} vs folds_v1={len(v1)}')
assert np.isfinite(X1).all(), 'ada NaN/Inf di embedding v1'

X    = X1[pos]                       # reindex: baris ke-i X = baris ke-i folds_v2
y    = v2['label'].to_numpy()
fold = v2['fold'].to_numpy()
assert len(X) == len(y) == len(fold) == len(v2)

# Sanity align: buktikan ulang lewat filepath di sampel acak (jangan percaya, verifikasi)
rng = np.random.default_rng(CFG.seed)
for i in rng.choice(len(v2), size=min(5, len(v2)), replace=False):
    fp = v2['filepath'].iloc[i]
    assert np.array_equal(X[i], X1[pos_of[fp]]), f'align gagal di baris {i} ({fp})'
print('✅ align terverifikasi')
print('X', X.shape, '| dist kelas y:', dict(zip(*np.unique(y, return_counts=True))))
print('fold unik:', sorted(np.unique(fold).tolist()))

---
## 🚀 CV — pakai kolom fold folds_v2 (tak di-stratify ulang)

In [ ]:
# Cell 5 — 5-fold CV kNN terkunci: train fold!=f, prediksi fold==f → OOF
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

K, LABELS = 15, [0, 1, 2]
folds = sorted(np.unique(fold).tolist())
oof = np.zeros((len(y), 3), dtype=float)
per_fold = []

for f in folds:
    tr, va = (fold != f), (fold == f)
    assert va.sum() > 0, f'fold {f} kosong'
    clf = KNeighborsClassifier(n_neighbors=K, metric='cosine', weights='distance')
    clf.fit(X[tr], y[tr])
    proba = clf.predict_proba(X[va])
    # Tempatkan kolom sesuai clf.classes_ (defensif kalau suatu kelas hilang di train fold)
    va_idx = np.where(va)[0]
    for j, c in enumerate(clf.classes_):
        oof[va_idx, int(c)] = proba[:, j]
    f1 = f1_score(y[va], oof[va].argmax(1), labels=LABELS, average='macro', zero_division=0.0)
    per_fold.append(f1)
    print(f'fold {f}: Macro-F1 = {f1:.4f}  (n_val={int(va.sum())})')

per_fold = np.array(per_fold)
oof_pred = oof.argmax(1)
oof_overall = f1_score(y, oof_pred, labels=LABELS, average='macro', zero_division=0.0)

print()
print(f'CV v2   mean={per_fold.mean():.4f}  min={per_fold.min():.4f}  std={per_fold.std():.4f}')
print(f'OOF overall Macro-F1 = {oof_overall:.4f}')

---
## 📊 BANDINGKAN v2 vs v1

In [ ]:
# Cell 6 — Verdict pakai aturan anti-overfit yang sama dgn probe_grid + cek per kelas
from sklearn.metrics import confusion_matrix

V1 = dict(mean=0.9901, min=0.9896, std=0.0005)   # referensi CV v1 (siglip2so400m + kNN, non-TTA)
m2, mn2, sd2 = per_fold.mean(), per_fold.min(), per_fold.std()
dmean, dmin = m2 - V1['mean'], mn2 - V1['min']

print(f"v1: mean={V1['mean']:.4f}  min={V1['min']:.4f}  std={V1['std']:.4f}")
print(f'v2: mean={m2:.4f}  min={mn2:.4f}  std={sd2:.4f}')
print(f'Δmean={dmean:+.4f}   Δmin={dmin:+.4f}\n')

if dmean > 0.002 and dmin >= 0:
    print('✅ v2 lebih baik (mean naik jelas, min tak turun) — cleaning membantu, PAKAI folds_v2')
elif abs(dmean) < 0.002:
    print('➖ Dalam noise (|Δmean|<0.002) — cleaning netral untuk skor.')
    print('   Pilih v2 kalau min≥ & std≤ v1 (data lebih bersih itu bonus); else pertahankan v1.')
elif dmean > 0 and dmin < 0:
    print('⚠️  mean naik tapi min turun — TOLAK, ini bukan gain nyata (satu fold beruntung)')
else:
    print('❌ v2 lebih buruk — kemungkinan cleaning membuang sampel sulit yg labelnya benar.')
    print('   Cek arah cleaning bareng Track A sebelum retrain multi-backbone.')

# Reviewer bilang Organic↔Recyclable saling tertukar — buktikan cleaning memperbaikinya
per_class = f1_score(y, oof_pred, labels=LABELS, average=None, zero_division=0.0)
print('\nPer-kelas F1 (0=Recyclable, 1=Electronic, 2=Organic):', np.round(per_class, 4))
print('Confusion matrix:')
print(confusion_matrix(y, oof_pred, labels=LABELS))

---
## 💾 HANDOFF — OOF v2 ke Track C (guard anti-overwrite)

In [ ]:
# Cell 7 — Simpan OOF v2 + meta. File existing = FileExistsError (bukan silent overwrite)
import json

OUT      = CFG.save_dir
oof_path  = os.path.join(OUT, 'oof_siglip2so400m_knn_v2.npy')
meta_path = os.path.join(OUT, 'oof_siglip2so400m_knn_v2_meta.json')

for p in (oof_path, meta_path):
    if os.path.exists(p):
        raise FileExistsError(f'{p} sudah ada — jangan overwrite. Hapus manual kalau memang mau regenerate.')

np.save(oof_path, oof)
meta = dict(
    combo='siglip2so400m', head='knn', k=15, metric='cosine', weights='distance', tta=False,
    folds='folds_v2.csv', seed=int(CFG.seed),
    n=int(len(y)), dropped_from_v1=int(n_drop), relabeled=int(relabeled),
    cv_mean=float(m2), cv_min=float(mn2), cv_std=float(sd2), oof_overall=float(oof_overall),
    label_order=[0, 1, 2],
    align_note='baris ke-i oof = baris ke-i folds_v2.csv (urutan sama). Track C join lewat urutan ini.',
)
with open(meta_path, 'w') as fh:
    json.dump(meta, fh, indent=2)

print('OOF v2 tersimpan:', oof_path, oof.shape)
print('meta          :', meta_path)
print('\nSIAP DISERAHKAN KE TRACK C — oof align dgn folds_v2.csv (urut baris sama).')
print('Umumkan ke grup: "OOF v2 hijau, CV v2 =', f'{m2:.4f}\"')

---
## Langkah berikutnya

1. **Catat verdict Cell 6.** Kalau v2 ≥ v1 (atau setara tapi lebih bersih) → semua retrain/ensemble berikut pakai `folds_v2.csv`.
2. **Serahkan `oof_siglip2so400m_knn_v2.npy` ke Track C** untuk threshold tuning + weight search (nested validation, sesuai Track C v3 plan). Track C align lewat urutan baris = folds_v2.
3. **Diagnostik drop/relabel per kelas (Cell 3 & 6)** masuk bagian *Data Cleaning* di report — itu cerita "diagnosis → bersihkan → CV naik" yang dinilai reviewer.
4. **Kalau mau ensemble multi-backbone v2 nanti:** ulangi Cell 4–5 untuk backbone lain (ganti `LOCKED`), tapi ingat concat sempat ditolak di v1 (dalam noise). Uji ulang di data bersih HANYA kalau sempat.

> **Belum kepakai di sini:** varian `_tta` (ditolak di v1) dan head selain kNN. Retrain ini sengaja fokus ke komposisi terkunci biar perbandingan v2-vs-v1 bersih (satu variabel berubah: datanya).